In [ ]:
#XGBoost
from xgboost import XGBRegressor
import pandas as pd
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

fold_rmse = []
    
model = XGBRegressor(
    n_estimators = 200,
    learning_rate = 0.05,
    max_depth = 4,
    random_state = 42
)

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
submission.to_csv('submission_xgboost.csv', index=False)


In [ ]:
#LGBMBoost
from lightgbm import LGBMRegressor
import pandas as pd
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

fold_rmse = []
    
model = LGBMRegressor(
        n_estimators = 200,
        learning_rate = 0.05,
        num_leaves = 15,
        max_depth = 4,
        random_state = 42,
        verbose = -1
    )

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
submission.to_csv('submission_lgbmboost.csv', index=False)

In [ ]:
#CatBoost
from catboost import CatBoostRegressor
import pandas as pd
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

fold_rmse = []
    
model = CatBoostRegressor(
        iterations = 200,
        learning_rate = 0.05,
        depth = 4,
        verbose = 0,
        random_state = 42
    )

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
submission.to_csv('submission_catboost.csv', index=False)

In [ ]:
#ElasticNet (best params)
from sklearn.linear_model import ElasticNet
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

model = TransformedTargetRegressor(
    regressor = create_pipe(
        ElasticNet(alpha = 0.01, l1_ratio = 0.5, max_iter = 20000),
        preprocessor,
        scale = True
    ),
    func = np.log1p,
    inverse_func = np.expm1
)

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
submission.to_csv('submission_elasticnet.csv', index=False)


In [ ]:
#CatBoost (best params)
from catboost import CatBoostRegressor
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

model = TransformedTargetRegressor(
    regressor = create_pipe(
        CatBoostRegressor(iterations = 500, depth = 6, learning_rate = 0.1, verbose = 0, random_state = 42),
        preprocessor,
        scale = False
    ),
    func = np.log1p,
    inverse_func = np.expm1
)

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
submission.to_csv('submission_catboost.csv', index=False)


In [ ]:
#XGBoost (best params)
from xgboost import XGBRegressor
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

model = TransformedTargetRegressor(
    regressor = create_pipe(
        XGBRegressor(n_estimators = 500, learning_rate = 0.1, max_depth = 3, colsample_bytree = 0.8, subsample = 0.8, random_state = 42, objective = 'reg:squarederror', eval_metric = 'rmse', n_jobs = 1),
        preprocessor,
        scale = False
    ),
    func = np.log1p,
    inverse_func = np.expm1
)

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
submission.to_csv('submission_xgboost.csv', index=False)


In [ ]:
#Lasso (best params)
from sklearn.linear_model import Lasso
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

model = TransformedTargetRegressor(
    regressor = create_pipe(
        Lasso(alpha = 0.01, random_state = 42, max_iter=20000),
        preprocessor,
        scale = True
    ),
    func = np.log1p,
    inverse_func = np.expm1
)

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
submission.to_csv('submission_lasso.csv', index=False)


In [ ]:
#LGBMBoost (best params)
from lightgbm import LGBMRegressor
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

model = TransformedTargetRegressor(
    regressor = create_pipe(
        LGBMRegressor(n_estimators = 500, learning_rate = 0.05, max_depth = 3, num_leaves = 7, random_state=42, verbose=-1, n_jobs=1),
        preprocessor,
        scale = False
    ),
    func = np.log1p,
    inverse_func = np.expm1
)

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
submission.to_csv('submission_lgbmboost.csv', index=False)


In [ ]:
#Ridge (best params)
from sklearn.linear_model import Ridge
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

model = TransformedTargetRegressor(
    regressor = create_pipe(
        Ridge(alpha = 100, random_state = 42, max_iter=20000),
        preprocessor,
        scale = True
    ),
    func = np.log1p,
    inverse_func = np.expm1
)

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
submission.to_csv('submission_ridge.csv', index=False)